# 23 - Pass 11b: how much of an "all headlines" accuracy is training data? (one commit, GPU)

**The question.** The seed paper's Table II reports 93.88% (GAN-BERT + MTL) and 92.04% (BERT) with support 28,619 for every model, i.e. over the
**whole dataset**, while its text says the test split is 5,724 items. This notebook trains the same kind of models and scores them the same way:
on the labeled rows L, the unlabeled rows U, dev, test, and on **all rows together**. Training is exactly the untagged runs of Passes 1 and 11
(tag `full`); only the scoring changes.

| Runs | What it shows |
|---|---|
| **A5** (the seed recipe) at 70% labels, 5 seeds | accuracy on all rows vs on the held-out test set; their 93.88 is the all-rows figure for this kind of model |
| **A1** (BERT) at 70% labels, 5 seeds | the same for plain BERT; U (never labeled) should behave like test, L like training |
| **A1** with all labels, 5 seeds | the seed paper's BERT setting (trained on the whole 80% split; their reported BERT: 92.04) |
| **A1 at 1%, seed 0** (tag `rerun11`) | exact-match check of Pass 1 after the data-loading change (the re-run check that notebook 22 did not include) |

**16 runs, about 2.5 GPU-hours (about 1.2 h on T4 x2; estimate).** Resumable: if the session stops, run the run cell again.
Also checks that every `full` run reproduces the original run's test probabilities exactly (training is deterministic; the extra scoring changes nothing).

**Pre-registration.** Predictions H11.24-H11.28 are in the addendum of `vault/Pass 11 Predictions.md`; the notebook refuses to start unless the file says FROZEN.

**Before you run:** Input = the same `gainsplit-processed` dataset as notebook 22 (no notebook-15 output needed: no MLM encoder is used).
**Internet: on**, **Accelerator: GPU T4 x2**, then **Save Version -> Save & Run All (Commit)**.

## 1. Get the code
Clones the public repo and prints the **commit**. Every run records it in its `result.json`.

In [ ]:
import os, subprocess, sys, time, json, shutil
from pathlib import Path
T_START = time.time()
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

## 2. Tests (about two minutes)
If anything fails, stop and paste the output.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, "tests failed: see output above"

## 3. Pre-registration check
The predictions (including the addendum for this notebook) must be committed and FROZEN before any run.

In [ ]:
REQUIRE_FROZEN = True
pred = Path(f"{REPO_DIR}/vault/Pass 11 Predictions.md").read_text(encoding="utf-8")
status = [l.split(":", 1)[1].strip() for l in pred.splitlines()[:8] if l.startswith("status:")]
print("predictions status:", status, "| addendum for notebook 23 present:", "H11.24" in pred)
if REQUIRE_FROZEN:
    assert status == ["FROZEN"] and "H11.24" in pred, "predictions (with the H11.24-28 addendum) must be FROZEN in this commit"

## 4. Data and the 70% label budget
Same as notebook 22: the 70% sets are drawn with the same procedure and seeds as `budgets.json`, checked against the frozen 30% sets, and read through `GAINSPLIT_EXTRA_BUDGETS`.

In [ ]:
import torch
from gainsplit.data import find_data_dir, verify, make_extra_budgets, load_splits, EXTRA_BUDGETS_ENV
DATA_DIR = find_data_dir("/kaggle/input")
verify(DATA_DIR)
print("data:", DATA_DIR, "- all files match manifest.json")
extra = make_extra_budgets(DATA_DIR, (70,))
Path("/kaggle/working/extra").mkdir(parents=True, exist_ok=True)
EXTRA_PATH = "/kaggle/working/extra/budgets_extra.json"
Path(EXTRA_PATH).write_text(json.dumps(extra))
os.environ[EXTRA_BUDGETS_ENV] = EXTRA_PATH
committed = json.loads(Path(f"{REPO_DIR}/results/pass11/budgets_extra_p70.json").read_text())
assert committed == json.loads(json.dumps(extra)), "the 70% sets differ from the ones notebook 22 used"
print("70% sets identical to notebook 22's; GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 5. The list of runs

In [ ]:
from gainsplit.plans import PLANS
PLAN = "pass11_full"
OUT_DIR = f"/kaggle/working/results/{PLAN}"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
configs = PLANS["pass11_full"]()
for c in configs:
    print(f"{c.run_id:22s} arm={c.arm:3s} labels={c.budget:3d}% seed={c.seed} eval_train={c.eval_train}")
print(len(configs), "runs")
for b, s in sorted({(c.budget, c.seed) for c in configs}):
    load_splits(DATA_DIR, b, s)

## Run
One run per GPU; when a GPU frees up, the next starts. Logs in `results/pass11_full/logs/<run_id>.log`. If the session stops halfway, run this cell again.

In [ ]:
from gainsplit.runner import run_all
table = run_all(configs, DATA_DIR, OUT_DIR)
table.to_csv(f"{OUT_DIR}/summary.csv", index=False)
print(f"total so far: {(time.time() - T_START) / 60:.1f} min")

## Analysis (CPU, about a minute)
Accuracy (0.5 threshold) of each `full` run on L, U, all training rows, dev, test and **all rows together**; mean and s.d. over seeds; the exact re-run check against
the originals in the committed results folders (Passes 1 and 11); and the descriptive checks H11.24-H11.28. The seed paper's figures: 93.88 (GAN-BERT + MTL), 92.04 (BERT).

In [ ]:
from gainsplit import analysis_pass11 as a11
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
RESULTS_DIRS = [d for d in [f"{REPO_DIR}/results/{p}" for p in ("pass1", "pass11")] if Path(d).exists()] + [OUT_DIR]
print("results folders:", RESULTS_DIRS)
tables = a11.run_full(RESULTS_DIRS, DATA_DIR, f"{OUT_DIR}/analysis")
for name in ("full_summary", "rerun_check", "full_verdicts"):
    print(f"\n=== {name}")
    display(tables[name])

## Download
Zips the results folder. A commit keeps it in the Output tab: download `results_pass11_full.zip` and send it with the notebook log.

In [ ]:
print(shutil.make_archive("/kaggle/working/results_pass11_full", "zip", OUT_DIR))
print(f"notebook time: {(time.time() - T_START) / 60:.1f} min")